In [1]:
import socket

In [2]:
import threading

In [3]:
import time

In [4]:
HOST = "127.0.0.1"
PORT = 65004
MAX_WORKERS = 2

In [5]:
worker_slots = threading.Semaphore(MAX_WORKERS)

In [6]:
def handle_client(conn,addr):
    with conn:
        request = conn.recv(1024).decode();
        print(f"[worker - {threading.get_ident()}] waiting for a free addr slot ({addr})")
        with worker_slots:
            print(f"[worker - {threading.get_ident()}] got a slot, working on {addr}")
            time.sleep(2)

            reply = f"Processed '{request}' by worker thread {threading.get_ident()}"
            conn.sendall(reply.encode())

            print(f"[worker - {threading.get_ident()}] done with {addr}, releasing solt")
        
    

In [7]:
def main():
    with socket.socket(socket.AF_INET,socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET,socket.SO_REUSEADDR,1)
        s.bind((HOST,PORT))
        s.listen(5)
        print(f"[disparther] listening on {HOST}:{PORT} (max {MAX_WORKERS} concurrent workers)")

        while True:
            conn,addr = s.accept()
            print(f"[dispather] accepted {addr}, spawning worker thread")

            worker = threading.Thread(
                target = handle_client,
                args = (conn,addr)
            )
            worker.start()
        

In [8]:
if __name__ == "__main__":
    main()

[disparther] listening on 127.0.0.1:65004 (max 2 concurrent workers)
[dispather] accepted ('127.0.0.1', 38664), spawning worker thread
[worker - 129498128299712] waiting for a free addr slot (('127.0.0.1', 38664))
[worker - 129498128299712] got a slot, working on ('127.0.0.1', 38664)
[dispather] accepted ('127.0.0.1', 38670), spawning worker thread
[worker - 129498119907008] waiting for a free addr slot (('127.0.0.1', 38670))
[worker - 129498119907008] got a slot, working on ('127.0.0.1', 38670)
[dispather] accepted ('127.0.0.1', 38678), spawning worker thread
[worker - 129498111514304] waiting for a free addr slot (('127.0.0.1', 38678))
[dispather] accepted ('127.0.0.1', 38688), spawning worker thread
[worker - 129497626048192] waiting for a free addr slot (('127.0.0.1', 38688))
[dispather] accepted ('127.0.0.1', 38696), spawning worker thread
[worker - 129497617655488] waiting for a free addr slot (('127.0.0.1', 38696))
[worker - 129498128299712] done with ('127.0.0.1', 38664), relea

KeyboardInterrupt: 